<a href="https://colab.research.google.com/github/SoumajyotiDhut/rare-disease-identification-system/blob/main/notebooks/colab/Day_7_ZebraMap_NLP_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive, userdata
import os, json, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datetime import datetime

drive.mount('/content/drive')

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✓ HF Token set")
except:
    print("⚠ No HF Token")

BASE    = "/content/drive/MyDrive/rare_disease_project"
DATA    = f"{BASE}/data"
RESULTS = f"{BASE}/results"
MODELS  = f"{BASE}/models"

os.makedirs(RESULTS, exist_ok=True)
os.makedirs(MODELS,  exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

# Load splits
with open(f"{DATA}/splits/tier_a_train.pkl", "rb") as f:
    tier_a_train = pickle.load(f)
with open(f"{DATA}/splits/tier_a_test.pkl", "rb") as f:
    tier_a_test = pickle.load(f)
with open(f"{DATA}/splits/tier_c_train.pkl", "rb") as f:
    tier_c_train = pickle.load(f)
with open(f"{DATA}/splits/tier_c_test.pkl", "rb") as f:
    tier_c_test = pickle.load(f)
with open(f"{DATA}/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print(f"\n✓ Splits loaded")
print(f"  Tier A train : {len(tier_a_train)} samples")
print(f"  Tier A test  : {len(tier_a_test)} samples")
print(f"  Tier C train : {len(tier_c_train)} samples")
print(f"  Tier C test  : {len(tier_c_test)} samples")

Mounted at /content/drive
✓ HF Token set
✓ Device: Tesla T4

✓ Splits loaded
  Tier A train : 10711 samples
  Tier A test  : 1941 samples
  Tier C train : 3759 samples
  Tier C test  : 967 samples


In [ ]:
import ast
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Load full CSV for label encoding
df = pd.read_csv(f"{DATA}/clean_multimodal_samples.csv")

# Fresh label encoder from disease_name
new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

# Add symptom_text if missing
def make_symptom_text(sym_str):
    try:
        syms = sym_str if isinstance(sym_str, list) \
                       else ast.literal_eval(sym_str)
        return ' [SEP] '.join([s.lower().strip() for s in syms])
    except:
        return str(sym_str)

if 'symptom_text' not in df.columns:
    df['symptom_text'] = df['symptoms'].apply(make_symptom_text)

# Use tier_a for common baseline
# Use tier_c for ultra-rare baseline
# Build combined for Exp 4 overall baseline
label_counts = df['label'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df_filtered  = df[df['label'].isin(valid_labels)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

# Add symptom_text to splits
train_df['symptom_text'] = train_df['symptoms'].apply(
    make_symptom_text)
test_df['symptom_text']  = test_df['symptoms'].apply(
    make_symptom_text)

print(f"✓ Data ready")
print(f"  Train : {len(train_df)}")
print(f"  Test  : {len(test_df)}")
print(f"  Classes: {NUM_CLASSES}")

✓ Data ready
  Train : 29049
  Test  : 7263
  Classes: 1199


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2")
print("✓ Tokenizer loaded")

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts      = df['symptom_text'].tolist()
        self.labels     = df['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(
                                self.labels[idx], dtype=torch.long)
        }

class BioBERTClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "dmis-lab/biobert-base-cased-v1.2")
        hidden = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

nlp_train_ds = SymptomDataset(train_df, tokenizer)
nlp_test_ds  = SymptomDataset(test_df,  tokenizer)

nlp_train_loader = DataLoader(nlp_train_ds, batch_size=32,
                               shuffle=True,  num_workers=2)
nlp_test_loader  = DataLoader(nlp_test_ds,  batch_size=32,
                               shuffle=False, num_workers=2)

NLP_EPOCHS  = 10
nlp_model   = BioBERTClassifier(NUM_CLASSES).to(device)
criterion   = nn.CrossEntropyLoss()
optimizer   = AdamW(nlp_model.parameters(),
                    lr=2e-5, weight_decay=0.01)
total_steps = len(nlp_train_loader) * NLP_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

nlp_losses = []
nlp_accs   = []

print(f"\nTraining NLP — {NLP_EPOCHS} epochs")
print(f"  Samples : {len(nlp_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(NLP_EPOCHS):
    nlp_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in nlp_train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = nlp_model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            nlp_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(nlp_train_loader)
    acc      = correct / total * 100
    nlp_losses.append(avg_loss)
    nlp_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{NLP_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ NLP training complete")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_topk(model, loader, device, is_cnn=False, k=5):
    model.eval()
    all_preds, all_labels = [], []
    top5_correct, total   = 0, 0

    with torch.no_grad():
        for batch in loader:
            if is_cnn:
                inputs = batch['image'].to(device)
                labels = batch['label'].to(device)
                logits = model(inputs)
            else:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['label'].to(device)
                logits         = model(input_ids, attention_mask)

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            k_actual = min(k, logits.size(1))
            topk     = logits.topk(k_actual, dim=1).indices
            for i, lbl in enumerate(labels):
                if lbl in topk[i]:
                    top5_correct += 1
            total += labels.size(0)

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels, all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "f1_weighted"  : round(f1_score(
                            all_labels, all_preds,
                            average='weighted',
                            zero_division=0)*100, 2),
        "top5_accuracy": round(top5_correct/total*100, 2),
        "total_samples": total
    }

print("Evaluating NLP...")
nlp_metrics = evaluate_topk(nlp_model, nlp_test_loader, device)

print("\n" + "=" * 55)
print("EXP 4 — NLP BASELINE (ZebraMap, no augmentation)")
print("=" * 55)
print(f"  Accuracy     : {nlp_metrics['accuracy']}%")
print(f"  F1 Macro     : {nlp_metrics['f1_macro']}%")
print(f"  F1 Weighted  : {nlp_metrics['f1_weighted']}%")
print(f"  Top-5 Acc    : {nlp_metrics['top5_accuracy']}%")
print(f"  Test samples : {nlp_metrics['total_samples']}")

torch.save({
    'model_state_dict': nlp_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : nlp_metrics
}, f"{MODELS}/exp4_nlp_baseline.pt")
print(f"✓ NLP model saved")

Evaluating NLP...

EXP 4 — NLP BASELINE (ZebraMap, no augmentation)
  Accuracy     : 16.56%
  F1 Macro     : 2.75%
  F1 Weighted  : 10.84%
  Top-5 Acc    : 35.98%
  Test samples : 7263
✓ NLP model saved


In [ ]:
import torchvision.models as models
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image

class ZebraMapImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.samples   = []
        self.transform = transform

        for _, row in df.iterrows():
            try:
                images = row['images']
                if not isinstance(images, list):
                    images = eval(images)
                for img_info in images:
                    path = img_info['path']
                    if os.path.exists(path):
                        self.samples.append({
                            'path' : path,
                            'label': row['label']
                        })
                        break
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            img = Image.open(sample['path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return {
            'image': img,
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Building CNN datasets...")
cnn_train_ds = ZebraMapImageDataset(train_df, train_transform)
cnn_test_ds  = ZebraMapImageDataset(test_df,  test_transform)

cnn_train_loader = DataLoader(cnn_train_ds, batch_size=32,
                               shuffle=True,  num_workers=2)
cnn_test_loader  = DataLoader(cnn_test_ds,  batch_size=32,
                               shuffle=False, num_workers=2)

print(f"✓ CNN DataLoaders ready")
print(f"  Train images  : {len(cnn_train_ds)}")
print(f"  Test images   : {len(cnn_test_ds)}")

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False
        self.features   = nn.Sequential(
            *list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

CNN_EPOCHS = 15
cnn_model  = ResNet50Classifier(NUM_CLASSES).to(device)
optimizer2 = AdamW(
    filter(lambda p: p.requires_grad, cnn_model.parameters()),
    lr=1e-4, weight_decay=0.01
)
scheduler2 = CosineAnnealingLR(
    optimizer2, T_max=CNN_EPOCHS, eta_min=1e-6)

cnn_losses = []
cnn_accs   = []

print(f"\nTraining CNN — {CNN_EPOCHS} epochs")
print(f"  Images  : {len(cnn_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in cnn_train_loader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer2.zero_grad()
        logits = cnn_model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            cnn_model.parameters(), 1.0)
        optimizer2.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler2.step()
    avg_loss = total_loss / len(cnn_train_loader)
    acc      = correct / total * 100
    cnn_losses.append(avg_loss)
    cnn_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{CNN_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ CNN training complete")

In [ ]:
import matplotlib.pyplot as plt

print("Evaluating CNN...")
cnn_metrics = evaluate_topk(
    cnn_model, cnn_test_loader, device, is_cnn=True)

print("\n" + "=" * 55)
print("EXP 4 — CNN BASELINE (ZebraMap, no augmentation)")
print("=" * 55)
print(f"  Accuracy     : {cnn_metrics['accuracy']}%")
print(f"  F1 Macro     : {cnn_metrics['f1_macro']}%")
print(f"  F1 Weighted  : {cnn_metrics['f1_weighted']}%")
print(f"  Top-5 Acc    : {cnn_metrics['top5_accuracy']}%")
print(f"  Test samples : {cnn_metrics['total_samples']}")

torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : cnn_metrics
}, f"{MODELS}/exp4_cnn_baseline.pt")
print(f"✓ CNN model saved")

# Plot curves
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].plot(nlp_losses, color='#D85A30',
               linewidth=2, marker='o', markersize=3)
axes[0,0].set_title('NLP Loss — ZebraMap Baseline')
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Loss')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(nlp_accs, color='#1D9E75',
               linewidth=2, marker='o', markersize=3)
axes[0,1].set_title('NLP Accuracy — ZebraMap Baseline')
axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(cnn_losses, color='#534AB7',
               linewidth=2, marker='o', markersize=3)
axes[1,0].set_title('CNN Loss — ZebraMap Baseline')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Loss')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(cnn_accs, color='#BA7517',
               linewidth=2, marker='o', markersize=3)
axes[1,1].set_title('CNN Accuracy — ZebraMap Baseline')
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Experiment 4 — ZebraMap Baseline (No Augmentation)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS}/day7_zebramap_baseline_curves.png",
            dpi=150, bbox_inches='tight')
plt.show()

# Update tracker
with open(f"{RESULTS}/experiment_tracker.json", "r") as f:
    tracker = json.load(f)

tracker['experiments']['exp4_zebramap_baseline']['status']      = "complete"
tracker['experiments']['exp4_zebramap_baseline']['nlp_metrics'] = nlp_metrics
tracker['experiments']['exp4_zebramap_baseline']['cnn_metrics'] = cnn_metrics
tracker['last_updated'] = str(datetime.now().date())

with open(f"{RESULTS}/experiment_tracker.json", "w") as f:
    json.dump(tracker, f, indent=2)

# Day 7 summary
summary = {
    "day"        : 7,
    "experiment" : "exp4_zebramap_baseline",
    "train_size" : len(train_df),
    "test_size"  : len(test_df),
    "nlp_metrics": nlp_metrics,
    "cnn_metrics": cnn_metrics,
    "status"     : "Day 7 complete"
}

with open(f"{RESULTS}/day7_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 55)
print("DAY 7 COMPLETE ✓")
print("=" * 55)
print(f"  NLP Baseline — Accuracy : {nlp_metrics['accuracy']}%")
print(f"  NLP Baseline — Top-5    : {nlp_metrics['top5_accuracy']}%")
print(f"  CNN Baseline — Accuracy : {cnn_metrics['accuracy']}%")
print(f"  CNN Baseline — Top-5    : {cnn_metrics['top5_accuracy']}%")
print(f"\n  Next → Day 8: Results dashboard + comparison table")

In [1]:
from google.colab import drive
import torch
import numpy as np
import os

drive.mount('/content/drive')

MODELS = "/content/drive/MyDrive/rare_disease_project/models"
model_path = f"{MODELS}/exp4_nlp_baseline.pt"

if os.path.exists(model_path):
    size = os.path.getsize(model_path) / (1024*1024)
    print(f"✓ Model exists: exp4_nlp_baseline.pt ({size:.1f} MB)")

    torch.serialization.add_safe_globals([np.ndarray])
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    print(f"✓ Loaded successfully")
    print(f"\n  Metrics:")
    for k, v in ckpt['metrics'].items():
        print(f"    {k}: {v}")
else:
    print(f"✗ NOT FOUND: {model_path}")

Mounted at /content/drive
✓ Model exists: exp4_nlp_baseline.pt (417.2 MB)
✓ Loaded successfully

  Metrics:
    accuracy: 16.56
    f1_macro: 2.75
    f1_weighted: 10.84
    top5_accuracy: 35.98
    total_samples: 7263
